# Zeiten messen und plotten

## Agenda

1. Zeitmessung
2. Entwicklung eines Timing-Tools
3. Das `timeit`-Modul
4. Diagramme zeichnen mit `matplotlib`
5. Zeitmessungs-Diagramme
6. Zeitmessungs-Beispiele
    - Listenzugriff
    - Lineare Suche
    - Binäre Suche
    - Insertion Sort (Sortieren durch Einfügen)
    - Collatz-Vermutung
7. Erkenntnisse

## Vergleich von Algorithmen anhand ihrer Laufzeit

Bevor wir mit der eigentlichen Zeitmessung beginnen, wollen wir anhand eines Beispiels zeigen, dass verschiedene Algorithmen zur Lösung desselben Problems unterschiedlich schnell sein können. Die Effizienz eines Algorithmus lässt sich häufig durch die benötigte Laufzeit vergleichen.

Im folgenden Beispiel berechnen wir die Quadratwurzel einer Zahl auf zwei verschiedene Arten: einmal durch dummes Probieren mittels einer Schleife und einmal mit dem iterativen Heron-Algorithmus. Anschließend können wir die Laufzeit beider Ansätze vergleichen.

In [35]:
# Variante 1: Dummes Wurzelziehen durch Probieren
# Wir probieren alle Zahlen von 0 bis zur Zahl aus, bis wir ein Ergebnis finden,
# dessen Quadrat größer oder gleich unserer Zahl ist
def sqrt_brute_force(n, epsilon=1e-4):
    x = 0
    while x * x < n:  
        x += epsilon 
    return x

# Variante 2: Heron-Algorithmus (Newton-Raphson-Verfahren)
# Ein effizienter iterativer Algorithmus zur Berechnung der Quadratwurzel
def sqrt_heron(n, epsilon=1e-10):
    if n < 0:
        return None
    if n == 0:
        return 0
    
    # Starten wir mit einer ersten Schätzung
    x = n / 2
    
    # Wiederholen wir die Iteration bis zur gewünschten Genauigkeit
    while abs(x * x - n) > epsilon:
        x = (x + n / x) / 2  # Heron-Formel
    
    return x

In [44]:
# Genauigkeit festlegen
epsilon = 1e-6
number = 10_001
result_brute = sqrt_brute_force(number, epsilon)
print(f"Brute Force Ergebnis: {result_brute}")

Brute Force Ergebnis: 100.0050008395283


In [45]:
result_heron = sqrt_heron(number, epsilon)
print(f"Heron-Algorithmus Ergebnis: {result_heron}")
print(f"Zum Vergleich: sqrt({number}) = {number ** 0.5}")

Heron-Algorithmus Ergebnis: 100.00499987500625
Zum Vergleich: sqrt(10001) = 100.00499987500625


> **Das Heron-Verfahren (Wurzelziehen)**
>
>**Historie:**
> Eine der ältesten Rechenvorschriften aufgeführt in der *Metrika* von Heron, aber schon früher in Babylonien verwendet. (Babylonier ~1800 v. Chr. / Heron ~100 n. Chr.).
>
>**Kern-Idee (Geometrie):**
>* Wir suchen die Seitenlänge eines Quadrats mit Fläche $n$.
>* Wir starten mit einem Rechteck (Seite $x$). Die andere Seite ist dann $n/x$.
>* Da die wahre Wurzel zwischen diesen beiden Werten liegt, nehmen wir deren **Mittelwert** als neue, bessere Schätzung.
>
>**Der Algorithmus:**
>1. **Startwert:** Wähle ein beliebiges $x$ (z. B. $n/2$).
>2. **Verbesserung:** Berechne $x_{neu} = \frac{x + n/x}{2}$.
>3. **Wiederholung:** Nutze $x_{neu}$ als neues $x$ und wiederhole Schritt 2, bis sich der Wert kaum noch ändert.
>
> ![](images/heron_iterations.png)

Gefühlt ist das Heron Verfahren schneller. Aber das müssen wir doch auch genauer messen können. Lassen Sie uns die Zeit mal wirklich stoppen.

## 1. Zeit messen

Das [`time`](https://docs.python.org/3/library/time.html#module-time) Modul enthält Funktionen, um die aktuelle Systemzeit zu ermitteln und zu interpretieren.

In [ ]:
import time
time.time()

In [ ]:
time.time() // (365 * 24 * 60 * 60)

In [ ]:
time.localtime(time.time())

Indem wir „Zeitstempel“ zu Beginn und am Ende nehmen, können wir die Laufzeit von Code messen:

In [ ]:
time.sleep(0.5)

## 2. Entwicklung eines Timing-Tools

Wir können eine Hilfsfunktion erstellen, um die Ausführungszeit einer übergebenen Funktion zu messen:

In [ ]:
def timeit(fn):
    pass

In [ ]:
sum(range(10_000))

In [ ]:
timeit(?)

Die Verwendung einer Lambda-Funktion ist eine Möglichkeit, aber damit sind wir darauf beschränkt, eine Funktion auf eine bestimmte Weise aufzurufen (z.B. ohne Argumente). Stattdessen können wir die Funktion `eval` nutzen, um einen beliebigen Python-Code als Zeichenkette auszuführen.

In [ ]:
eval('1 + 2')

In [ ]:
eval('sum(range(10_000))')

In [ ]:
def timeit(stmt='pass'):
    pass

In [ ]:
timeit('sum(range(10_000))')

Um die Zeitmessung stabiler zu machen, können wir Zeitmessung mehrmals ausführen und einen Mittelwert zurückgeben:

In [ ]:
def timeit(stmt='pass', number=1):
    pass

In [ ]:
timeit('sum(range(10_000))', number=1000)

## 3. Das `timeit`-Modul

Das [`timeit`](https://docs.python.org/3/library/timeit.html) Modul ist eine eingebaute Bibliothek, mit der sich die Ausführungszeit von als Zeichenkette übergebenem Code messen lässt.

In [ ]:
import timeit
timeit.timeit('sum(range(10_000))', number=1000)

Es unterstützt außerdem das Übergeben von „Setup“-Code, der nicht mitgemessen wird.

In [ ]:
timeit.timeit('sum(r)', 
              setup='r = range(10_000)', 
              number=1000)

Damit und mit einer List-Comprehension können wir sehr einfach Zeiten für eine Spanne von Eingabewerten berechnen:

In [ ]:
[timeit.timeit('sum(r)',
               setup=f'r = range({n})',
               number=1000)
 for n in range(1000, 11_000, 1000)]

Manchmal möchten wir Funktionen, die im Notebook definiert wurden, im an `timeit` übergebenen Code verwenden. Dafür müssen wir das Argument `globals` nutzen:

In [ ]:
def fib(n):
    if n == 0:
        return 0
    elif n == 1:
        return 1
    else:
        return fib(n-1) + fib(n-2)

In [ ]:
[fib(n) for n in range(15)]

In [ ]:
[timeit.timeit(f'fib({n})',
               number=1000,
               globals=globals()) # recall: "globals()" returns a dictionary of everything
                                  # defined in this module; timeit needs it to access `fib`
 for n in range(1, 15)]

### Magische Timing-Befehle

Es gibt auch „magische“ Notebook-Befehle, mit denen sich Code messen lässt – sie heißen `%time`, `%timeit` und `%%timeit`.

In [ ]:
%time sum(range(10_000))

In [ ]:
%timeit -n 1000 sum(range(10_000))

In [ ]:
%%timeit -n 1000

total = 0
for n in range(10_000):
    total += n

Es ist weniger komfortabel, die magischen Befehle zu nutzen, wenn man die Messergebnisse in einer Datenstruktur (z.B. Liste) speichern und weiterverarbeiten möchte. Deshalb verwenden wir meist das `timeit`-Modul.
Probieren wir das jetzt nochmal an unseren ursprünglichen Funktionen aus:

In [ ]:
%timeit -n 1000 sum_with_loop(range(10_000))

Jetzt können wir Zeiten messen. Interessant ist aber nicht nur der Vergleich von verschiedenen Algorithmen gegeneinander, sondern auch wie sich die Laufzeit eines Algorithmus abhängig von der Größe der zu verarbeitenden Daten ändert. Um das besser vergleichen zu können, benötigen wir noch ein wenig Werkzeug.

## 4. Zeichnen von Diagrammen mit `matplotlib`

Die [matplotlib-Bibliothek](https://matplotlib.org) unterstützt die Erstellung aller Arten von Visualisierungen. Wir werden sie zum Zeichnen einfacher zweidimensionaler Diagramme verwenden.Die wichtigste Plot-Funktion, die wir verwenden werden, ist `matplotlib.pyplot.plot` ([vollständige Dokumentation hier](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.plot.html#matplotlib.pyplot.plot)), die, wenn ihr zwei „array-ähnliche” Objekte gleicher Länge übergeben werden, deren Inhalt als x- und y-Achsenkoordinaten interpretiert und zeichnet. Wir werden in der Regel Tupel, Listen und Bereiche als array-ähnliche Objekte verwenden. Beachten Sie, dass Generatoren von matplotlib *nicht* als array-ähnlich betrachtet werden.

Schauen wir uns einige Beispiele an:

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import math

%matplotlib inline
plt.rcParams['figure.figsize'] = [10, 6] # set size of plot

In [ ]:
# das Semikolon am Ende unterdrückt die Ausgabe der Plot-Objekte
plt.plot([1, 2, 3, 4, 5], [50, 20, 30, 10, 40]);

In [ ]:
xs = range(11)
ys = [x*2 for x in xs]
plt.plot(xs, ys);

Wir können auch eine optionale Formatzeichenfolge für `plot` bereitstellen, die Markierungen, Linienstile und Farben für die Darstellung steuert.

Hier ist eine gekürzte Liste der Optionen, die aus der [vollständigen Dokumentation zu `plot`](https://matplotlib.org/api/_as_gen/matplotlib.pyplot.plot.html#matplotlib.pyplot.plot) kopiert wurde:

**Markierungen**

  - `.` : Punktmarkierung
  - `o` : Kreismarkierung
  - `s` : Quadratmarkierung
  - `d` : Rautenmarkierung

**Linienstile**
 
  - `-` : durchgezogene Linie
  - `--` : gestrichelte Linie
  - `:` : gepunktete Linie

**Farben**

  - `k` : schwarz
  - `r` : rot
  - `g` : blau
  - `b` : grün
  - `y` : gelb
  - `c` : cyan

Hier sind die oben genannten Diagramme mit einigen Farben und Stilen (wenn wir einen Linienstil weglassen, wird keine Verbindungslinie zwischen den Datenpunkten gezeichnet):

In [ ]:
plt.plot([1, 2, 3, 4, 5], [50, 20, 30, 10, 40], 'o--r');

In [ ]:
xs = range(11)
ys = [x*2 for x in xs]
plt.plot(xs, ys, 'dg');

Anstelle von normalen `range`-Objekten, die nur ganzzahlige Start-/Stopp-/Schrittwerte erlauben, bevorzugen wir mit matplotlib meistens die Funktionen `arange` und `linspace` aus der [numpy-Bibliothek](https://numpy.org).  
`arange` funktioniert ähnlich wie `range`, erlaubt aber auch Fließkommawerte für Start, Stopp und Schritt.  
Mit `linspace` können wir Start- und Stoppwerte (beide inklusive) sowie die Anzahl der zurückgegebenen Werte im Intervall angeben.

Beispiele für Aufrufe von `arange` und `linspace` (beachte, dass beide Funktionen numpy-Arrays zurückgeben, die iterierbar sind und an `plot` übergeben werden können):

In [ ]:
np.arange(0.5, 2.5, 0.1)

In [ ]:
np.linspace(10, 20, 41)

In [ ]:
np.linspace(1, 100_000, 50, dtype=int) # we can specify the data type to coerce values into integers

`plot` kann mehrmals in derselben Zelle aufgerufen werden, um mehrere Linien im selben Diagramm darzustellen. (Das liegt daran, dass sich matplotlib teilweise an der **Grammar of Graphics**, einem Konzept, das von Leland Wilkinson entwickelt wurde, orientiert.) Unten nutzen wir diese Möglichkeit zusammen mit `linspace` und einigen List Comprehensions, um einige typische Funktionen zur Abschätzung der Laufzeitkomplexität (dazu später mehr) über ein kleines Intervall zu zeichnen:

In [ ]:
count = 100
xs = np.linspace(0.1, 4, count)
ys_const        = [1] * count
ys_log          = [math.log(x) for x in xs]
ys_linear       = [x for x in xs]
ys_linearithmic = [x * math.log(x) for x in xs]
ys_quadratic    = [x**2 for x in xs]
ys_exponential  = [2**x for x in xs]

plt.plot(xs, ys_const, 'c')
plt.plot(xs, ys_log, 'r')
plt.plot(xs, ys_linear, 'b')
plt.plot(xs, ys_linearithmic, 'g')
plt.plot(xs, ys_quadratic, 'y');
plt.plot(xs, ys_exponential, 'm');

## 5. Zeitmessungen visualisieren

Das Darstellen von gemessenen Laufzeiten verschiedener Funktionen kann uns helfen, ein Gefühl dafür zu bekommen, wie deren Laufzeit mit zunehmender Eingabemenge wächst.

In [ ]:
# runtimes for sum for increasing sizes of input

ns = np.linspace(10, 10_000, 50, dtype=int)
ts = [timeit.timeit(f'sum(range({n}))', number=100)
      for n in ns]

plt.plot(ns, ts, 'or');

Offensichtlich ist die Laufzeit von `sum` direkt proportional zur Anzahl der Werte, auf die sie angewendet wird.

Wenn wir von einem linearen Zusammenhang ausgehen, können wir die durchschnittliche Steigung zwischen benachbarten Datenpunkten berechnen. Dadurch erhalten wir eine ungefähre Ausgleichslinie, mit der sich die Laufzeit von `sum` vorhersagen lässt.

In [ ]:
total = 0

for i in range(len(ns)-1):
    total += ? # recall: slope = rise / run
    
avg_slope = total / (len(ns)-1)

In [ ]:
avg_slope

Mithilfe des `statistics` Paketes können wir auch den Mittelwert berechnen und als unsere durchschnittliche Steigung festlegen.

In [ ]:
import statistics
avg_slope = statistics.mean((ts[i+1] - ts[i]) / (ns[i+1] - ns[i]) for i in range(len(ns)-1))
avg_slope

Es ist nachvollziehbar, dass wir für unsere linearen Abschätzungsfunktionen einfach $c_1$ < Steigung und $c_2$ > Steigung wählen müssen. Zur Sicherheit verwenden wir in unserer Grafik unten die Faktoren 0,8 und 1,2 (und zusätzlich zeichnen wir die Linie mit der unveränderten Steigung).:

In [ ]:
plt.plot(ns, ts, 'or')
plt.plot(ns, [avg_slope*n for n in ns], '--b');

In [ ]:
# i.e., for input of size N, runtime is estimated at:
for n in np.linspace(1, 100_000_000, 11, dtype=int):
    print(f'Runtime of sum(range({n:>11,})) ~ {avg_slope * n / 100:>5.2f} s')

Wenn uns das zu ungenau ist, können wir sogar [`polyfit`](https://numpy.org/doc/1.18/reference/generated/numpy.polyfit.html) verwenden, um eine am besten passende Polynomfunktion beliebigen Grades für unsere Daten zu berechnen:

In [ ]:
degree = 10
coeffs = np.polyfit(ns, ts, degree)
p = np.poly1d(coeffs)
plt.plot(ns, ts, 'or')
plt.plot(ns, [p(n) for n in ns], '-b');

Sieht doch eigentlich ganz gut aus. So können wir die Laufzeiten doch schon abschätzen. Oder?

In [ ]:
# i.e., for input of size N, runtime is estimated at:
for n in np.linspace(1, 100_000_000, 11, dtype=int):
    print(f'Runtime of sum(range({n:>11,})) ~ {p(n)/100:.2f} s')

Die Wahl einer ungeeigneten Funktion kann zu völlig falschen Vorhersagen der Laufzeit führen. Noch schlimmer: Die Ungenauigkeiten verstärken sich, je größer die Eingabe wird!

Wie wissen wir, welche Art von Funktion (z. B. logarithmisch, linear, Polynom n-ten Grades, exponentiell) sich zum Modellieren des Laufzeitverhaltens von Algorithmen eignet?

Können wir das zuverlässig durch empirische Beobachtung herausfinden?


Die Antwort ist **NEIN**. Und deswegen machen wir jetzt, bevor wir uns weiteren Zeitmessungen zuwenden ein wenig "Mathematik".

## 6. Weitere Zeitmessungen

### Built-in list indexing

Welche Laufzeitkomplexität hat das eingebaute Indizieren auf Listen?

In [ ]:
lst = list(range(1_000_000))
ns = np.linspace(0, len(lst), 1000, endpoint=False, dtype=int)
ts = [timeit.timeit(f'_ = lst[{n}]',
                    globals=globals(), 
                    number=10000) 
      for n in ns]

plt.plot(ns, ts, 'or');

Beobachtung: Das Zugreifen auf ein Element einer Liste per Index dauert immer gleich lange – unabhängig davon, wo sich das Element in der Liste befindet.

Warum? **Eine Python-Liste basiert intern auf einem Array.** Jeder „Platz“ im Array ist eine Referenz (also eine Adresse mit fester Länge) auf ein Objekt. Um ein Element an einem bestimmten Index zu erreichen, macht der zugrunde liegende Code:

1. Berechnet den *Offset* im Array, indem der Index mit der Größe einer Referenz multipliziert wird
2. Addiert den berechneten Offset zur *Basisadresse* des Arrays, so erhält man die Adresse der Referenz
3. Greift auf die Referenz zu und lädt damit das zugehörige Element

Jeder dieser Schritte kann in konstanter Zeit ausgeführt werden.

![](images/array_indexing.png)

### Lineare Suche

Wie verhält sich die Laufzeit beim Suchen eines Elements in einer unsortierten Liste?

In [ ]:
def contains(lst, x):
    pass

In [ ]:
import random
lst = list(range(100))
random.shuffle(lst)

contains(lst, 10)

In [ ]:
# runtimes when searching for a present element in a randomly shuffled list

ns = np.linspace(10, 10_000, 100, dtype=int)
ts = [timeit.timeit('contains(lst, 0)', 
                    setup=f'lst=list(range({n})); random.shuffle(lst)',
                    globals=globals(),
                    number=100)
      for n in ns]

plt.plot(ns, ts, 'or');

In [ ]:
# runtimes when searching for an element that is not present

ns = np.linspace(1_000, 10_000, 100, dtype=int)
ts = [timeit.timeit('contains(lst, -1)', 
                    setup=f'lst=list(range({n}))',
                    globals=globals(),
                    number=100)
      for n in ns]

plt.plot(ns, ts, 'or');

### Binäre Suche

Wie verhält sich die Laufzeit bei der binären Suchen eines Elements in einer sortierten Liste?

In [ ]:
def contains(lst, x):
    pass

In [ ]:
lst = list(range(1000))
contains(lst, 10)

In [ ]:
# runtimes when searching for different values in a fixed-size list

lst = list(range(1000))
ns = range(1000)
ts = [timeit.timeit(stmt=f'contains(lst, {x})', 
                    globals=globals(), 
                    number=1000)
      for x in range(1000)]

plt.plot(ns, ts, 'or');

In [ ]:
# runtimes when searching for an edge-value in lists of increasing size

ns = np.linspace(10, 10_000, 100, dtype=int)
ts = [timeit.timeit('contains(lst, 0)', 
                    setup=f'lst=list(range({n}))',
                    globals=globals(),
                    number=1000)
      for n in ns]

plt.plot(ns, ts, 'or');

### Insertion Sort (Sortieren durch Einfügen)

Wie ist die Laufzeit beim Sortieren durch Einfügen?

In [ ]:
def insertion_sort(lst):
    pass

In [ ]:
import random
lst = list(range(1000))
random.shuffle(lst)
plt.plot(lst, 'og');

In [ ]:
insertion_sort(lst)
plt.plot(lst, 'og');

In [ ]:
# runtimes for a randomized list

ns = np.linspace(100, 2000, 15, dtype=int)
ts = [timeit.timeit('insertion_sort(lst)',
                    setup=f'lst=list(range({n})); random.shuffle(lst)',
                    globals=globals(),
                    number=1)
         for n in ns]

plt.plot(ns, ts, 'or');

In [ ]:
# runtimes for an already sorted list

ns = np.linspace(100, 2000, 15, dtype=int)
ts = [timeit.timeit('insertion_sort(lst)',
                    setup=f'lst=list(range({n}))',
                    globals=globals(),
                    number=1)
         for n in ns]

plt.plot(ns, ts, 'or');

In [ ]:
# runtimes for a reversed list

ns = np.linspace(100, 2000, 15, dtype=int)
ts = [timeit.timeit('insertion_sort(lst)',
                    setup=f'lst=list(reversed(range({n})))',
                    globals=globals(),
                    number=1)
         for n in ns]

plt.plot(ns, ts, 'or');

In [ ]:
# above runtimes superimposed

ns = np.linspace(100, 2000, 15, dtype=int)
ts1 = [timeit.timeit('insertion_sort(lst)',
                     setup=f'lst=list((range({n})))',
                     globals=globals(),
                     number=1)
       for n in ns]
ts2 = [timeit.timeit('insertion_sort(lst)',
                     setup=f'lst=list(range({n})); random.shuffle(lst)',
                     globals=globals(),
                     number=1)
       for n in ns]

ts3 = [timeit.timeit('insertion_sort(lst)',
                     setup=f'lst=list(reversed(range({n})))',
                     globals=globals(),
                     number=1)
       for n in ns]

plt.plot(ns, ts1, 'og');
plt.plot(ns, ts2, 'ob');
plt.plot(ns, ts3, 'or');

### Collatz Vermutung

Die Collatz-Vermutung definiert eine Zahlenreihe, beginnend mit einer beliebigen positiven ganzen Zahl $n$. Die weiteren Werte der Reihe werden mit folgender Funktion berechnet:

$$f(n) = \begin{cases} n/2 &\text{falls $n$ gerade ist} \\ 3n+1 & \text{falls $n$ ungerade ist} \end{cases}$$

Die Vermutung besagt, dass die Reihe – egal mit welcher Startzahl – immer bei 1 endet.

Wie verhält sich die Laufzeit der Collatz-Folgen-Funktion, wenn die Werte von $n$ größer werden?

In [ ]:
def collatz(n):
    pass

In [ ]:
collatz(9)

In [ ]:
# runtimes for different values of n

ns = np.linspace(1, 100_000, 200, dtype=int)
ts = [timeit.timeit(f'collatz({n})',
                    globals=globals(),
                    number=100)
      for n in ns]

plt.plot(ns, ts, 'or');

Proving the conjecture is an open research problem! (I.e., it's possible that the series doesn't terminate for some value of $n$, though such $n$ is not known to exist.)

## 7. Erkenntnisse

- Mit Zeitmessungs- und Visualisierungsbibliotheken können wir das Laufzeitverhalten von Algorithmen systematisch messen und darstellen.
- Verschiedene Eigenschaften der Eingabe (z. B. zufällig, sortiert, umgekehrt) können einen großen Einfluss auf die Laufzeit von Algorithmen haben.
- Empirische Messungen der Laufzeit liefern nicht immer ein klares, genaues oder konsistentes Bild vom langfristigen Laufzeitverhalten einer Funktion.
- Die Wahl einer falschen Funktionsklasse zur Beschreibung des Laufzeitverhaltens eines Algorithmus kann zu völlig falschen Vorhersagen führen.
- Zeitmessungen sind hilfreich, aber wir brauchen eine systematischere und strengere Methode, um das Laufzeitverhalten von Algorithmen zu beschreiben und zu vergleichen!